# 🧠 Sesión 6 · Elegir entre varias opciones

Ya sabemos predecir **un número** y **sí/no**. Ahora la red elegirá entre **3 opciones**.

**Objetivo:** reconocer el **tipo de flor** (Iris) por el tamaño de sus pétalos y sépalos. Es el mismo conjunto de datos que usamos con scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.datasets import load_iris

## 1. Los datos

In [ ]:
iris = load_iris(as_frame=True)
datos = iris.frame
datos.head()

La columna `target` es el tipo de flor:
- **0** = setosa
- **1** = versicolor
- **2** = virginica

In [ ]:
nombres = ["setosa", "versicolor", "virginica"]
datos["target"].value_counts()

## 2. Preparar los datos

In [ ]:
X = datos.drop(columns="target")      # las 4 medidas
y = datos["target"]                    # el tipo de flor
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

## 3. Crear la red
Cambios para **varias opciones**:

| | Sí/no | Varias opciones |
|---|---|---|
| Última capa | `Dense(1, activation="sigmoid")` | `Dense(3, activation="softmax")` |
| loss | `"binary_crossentropy"` | `"sparse_categorical_crossentropy"` |

💡 La última capa tiene **3 neuronas**, una por cada flor. `softmax` reparte el **100 %** entre las 3 opciones.

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(4,)),                          # 4 medidas
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(3, activation="softmax")       # 3 tipos de flor
])
modelo.compile(loss="sparse_categorical_crossentropy",
               optimizer=keras.optimizers.Adam(0.01),
               metrics=["accuracy"])

## 4. Entrenar y examinar

In [ ]:
modelo.fit(X_train, y_train, epochs=100, verbose=0)

perdida, aciertos = modelo.evaluate(X_test, y_test, verbose=0)
print("Aciertos en el examen:", round(aciertos * 100), "%")

## 5. Usar la red
Medimos una flor nueva. La red nos da el **% de cada opción**.

In [ ]:
flor = pd.DataFrame([[5.0, 3.4, 1.5, 0.2]], columns=X.columns)
porcentajes = modelo.predict(flor, verbose=0)[0]

for nombre, p in zip(nombres, porcentajes):
    print(f"{nombre:12s} {p*100:5.1f} %")

print("👉 Es una", nombres[np.argmax(porcentajes)])

💡 `np.argmax` busca la opción con el **% más alto**.

Lo vemos en una gráfica de barras:

In [ ]:
plt.bar(nombres, porcentajes)
plt.ylabel("Probabilidad")
plt.show()

## 📝 Resumen: los 3 tipos de problema

| Quiero predecir... | Última capa | loss |
|---|---|---|
| Un **número** | `Dense(1)` | `"mean_squared_error"` |
| **Sí / no** | `Dense(1, activation="sigmoid")` | `"binary_crossentropy"` |
| **Varias opciones** | `Dense(nº opciones, activation="softmax")` | `"sparse_categorical_crossentropy"` |

📌 ¡Esta tabla es la más importante del curso!